# 🥇 NYC Yellow Taxi Analytics — Silver to Gold Semantics
---
### 👤 Developer Profile
* **Name:** `cacciari@gmail.com`
* **Contact:** [cacciari@gmail.com](mailto:cacciari@gmail.com)
* **LinkedIn:** [linkedin.com/in/cacciari](https://linkedin.com/in/cacciari)

### 📋 Module Overview
* **File Name:** `04_silver_to_gold`
* **Target Layer:** `Gold Layer (Consumer Facing / BI)`
* **Short Description:** High-performance analytical processing module generating multi-dimensional business matrices. Translates data into consumer-facing schemas, including specialized revenue metrics and window-function metrics evaluating user behavior and hourly density.

In [0]:
%skip
%pip install datacompy
%pip install datacompy[spark]

In [0]:
# %restart_python
%load_ext autoreload
%autoreload 2

#### Imports

In [0]:
# import pyspark.sql.functions as F
# import pyspark.sql.types as T
from typing import Dict, List, Any
from constants import BRONZE_TABLE, SILVER_TABLE, GOLD_TABLE_MONTHLY, GOLD_TABLE_WEEKLY, GOLD_TABLE_DAILY, GOLD_TABLE_HOURLY, GOLD_TABLE_MAY_ANALYTICS, AUDIT_COLUMNS
from modules.data_ingestion import *


#### Functions

#### Pipeline configuration

In [0]:
AUDIT_COLUMNS["AUD_ING_SLV_TO_GLD_SCRIPT"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split('/')[-1]

# WIDGET
dbutils.widgets.text("GOLD_TABLE_MONTHLY", GOLD_TABLE_MONTHLY)
dbutils.widgets.text("GOLD_TABLE_WEEKLY", GOLD_TABLE_WEEKLY)
dbutils.widgets.text("GOLD_TABLE_DAILY", GOLD_TABLE_DAILY)
dbutils.widgets.text("GOLD_TABLE_HOURLY", GOLD_TABLE_HOURLY)


#### Pre-Silver Ingestion Checks
* Show ingestion parameters

In [0]:
print("Parameter Set:")
print(f"\tSilver Table: {SILVER_TABLE}")
print(f"\tGold Table: {GOLD_TABLE_MONTHLY}")
print(f"\tGold Table: {GOLD_TABLE_WEEKLY}")
print(f"\tGold Table: {GOLD_TABLE_DAILY}")
print(f"\tGold Table: {GOLD_TABLE_HOURLY}")

print (f"Starting Transformation from {SILVER_TABLE} to {GOLD_TABLE_MONTHLY}")

try:
    print(f"\nAcquiring Data...:")
    df_silver = spark.table(SILVER_TABLE)
    print(f"\t{SILVER_TABLE} acquired. Total records: {df_silver.count()}")

except:
    print(f"\t{SILVER_TABLE} not found.")
    raise Exception(f"\t{SILVER_TABLE} not found.")

df_dict = {}


In [0]:
def build_gold_agg(source_table: str, 
                   target_table: str,
                   time_slice: str, 
                   verbose: bool=True) -> DataFrame:
    """
    Builds the gold table aggregations.
    
    Parameters
    ----------
    df_silver : DataFrame
        The silver table.
    target_table : str
        The target table.
    time_slice : str
        The time slice.
    verbose : bool, optional
        Whether to print the results, by default True

    """
    
    valid_slices = ['month', 'day', 'hour']
    if time_slice not in valid_slices:
        raise ValueError(f"Invalid time slice: {time_slice}. Valid slices are: {valid_slices}")

    df_silver = spark.table(source_table)
    
    df_aggregated = df_silver \
        .withColumn("TIME_SLICE", F.expr(f"{time_slice}(tpep_pickup_datetime)")) \
        .groupBy("TIME_SLICE") \
        .agg(
            # DIMENSION 1: OPERATIONAL & VOLUMETRICS
            F.count("*").alias("total_trips"),
            F.sum("passenger_count").alias("total_passenger_count"),
            F.avg("passenger_count").alias("avg_passenger_count"),

            # DIMENSION 2: FINANCIAL METRICS (MAIN REVENUE)
            # Total Amount (Final Revenue)
            F.sum("total_amount").alias("total_amount"),
            F.avg("total_amount").alias("avg_total_amount"),
            F.max("total_amount").alias("max_total_amount"),
            F.min("total_amount").alias("min_total_amount"),
            
            # Fare Amount (Base Fare)
            F.sum("fare_amount").alias("total_fare_amount"),
            F.avg("fare_amount").alias("avg_fare_amount"),
            F.max("fare_amount").alias("max_fare_amount"),
            F.min("fare_amount").alias("min_fare_amount"),
            
            # Tip Amount (Driver Tips)
            F.sum("tip_amount").alias("total_tip_amount"),
            F.avg("tip_amount").alias("avg_tip_amount"),
            F.max("tip_amount").alias("max_tip_amount"),
            F.min("tip_amount").alias("min_tip_amount"),

            # DIMENSION 3: SURCHARGES & ADDITIONAL FEES
            # Airport Fee
            F.sum("airport_fee").alias("total_airport_fee"),
            F.avg("airport_fee").alias("avg_airport_fee"),
            F.max("airport_fee").alias("max_airport_fee"),
            F.min("airport_fee").alias("min_airport_fee"),
            
            # Congestion Surcharge
            F.sum("congestion_surcharge").alias("total_congestion_surcharge"),
            F.avg("congestion_surcharge").alias("avg_congestion_surcharge"),
            F.max("congestion_surcharge").alias("max_congestion_surcharge"),
            F.min("congestion_surcharge").alias("min_congestion_surcharge"),
            
            # Improvement Surcharge
            F.sum("improvement_surcharge").alias("total_improvement_surcharge"),
            F.avg("improvement_surcharge").alias("avg_improvement_surcharge"),
            F.max("improvement_surcharge").alias("max_improvement_surcharge"),
            F.min("improvement_surcharge").alias("min_improvement_surcharge"),

            # DIMENSION 4: TRIP METRICS (DISTANCE & TIME)
            # Trip Distance
            F.sum("trip_distance").alias("total_trip_distance"),
            F.avg("trip_distance").alias("avg_trip_distance"),
            F.max("trip_distance").alias("max_trip_distance"),
            F.min("trip_distance").alias("min_trip_distance"),
            
            # Travel Time (Corrected to Seconds to avoid datediff zeroed results)
            F.sum(F.expr("unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)")).alias("total_travel_time_seconds"),
            F.avg(F.expr("unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)")).alias("avg_travel_time_seconds"),
            F.max(F.expr("unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)")).alias("max_travel_time_seconds"),
            F.min(F.expr("unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)")).alias("min_travel_time_seconds")
        )
        
    print(f"Aggregated by {time_slice}")
    return df_aggregated  
    





In [0]:
def build_gold_may_analytics(source_table: str, target_table: str, verbose: bool = True) -> DataFrame:
    """
    DEDICATED & ISOLATED function for May Gold analytics.
    Generates a multi-dimensional grain (Month, Day, Hour) with all metrics.
    """
    if verbose:
        print(f"🚀 [ISOLATED ENGINE] Starting exclusive processing for May: {source_table} -> {target_table}")
    
    # 1. Read from Silver Single Source of Truth
    df_silver = spark.table(source_table)
    
    # 2. Schema-safe filtering: Retain ONLY May records (Month = 5)
    df_only_may = df_silver.filter(F.month("tpep_pickup_datetime") == 5)
    
    # 3. Extract time-slice dimensions on the same row grain
    df_with_dimensions = df_only_may \
        .withColumn("MONTH_SLICE", F.month("tpep_pickup_datetime")) \
        .withColumn("DAY_SLICE", F.dayofmonth("tpep_pickup_datetime")) \
        .withColumn("HOUR_SLICE", F.hour("tpep_pickup_datetime"))
        
    # 4. Multi-dimensional aggregation based on the new timeline grain
    df_aggregated = df_with_dimensions.groupBy("MONTH_SLICE", "DAY_SLICE", "HOUR_SLICE").agg(
        # DIMENSION 1: OPERATIONAL & VOLUMETRICS
        F.count("*").alias("total_trips"),
        F.sum("passenger_count").alias("total_passenger_count"),
        F.avg("passenger_count").alias("avg_passenger_count"),

        # DIMENSION 2: FINANCIAL METRICS (MAIN REVENUE)
        F.sum("total_amount").alias("total_amount"),
        F.avg("total_amount").alias("avg_total_amount"),
        F.max("total_amount").alias("max_total_amount"),
        F.min("total_amount").alias("min_total_amount"),
        
        F.sum("fare_amount").alias("total_fare_amount"),
        F.avg("fare_amount").alias("avg_fare_amount"),
        F.max("fare_amount").alias("max_fare_amount"),
        F.min("fare_amount").alias("min_fare_amount"),
        
        F.sum("tip_amount").alias("total_tip_amount"),
        F.avg("tip_amount").alias("avg_tip_amount"),
        F.max("tip_amount").alias("max_tip_amount"),
        F.min("tip_amount").alias("min_tip_amount"),

        # DIMENSION 3: SURCHARGES & ADDITIONAL FEES
        F.sum("airport_fee").alias("total_airport_fee"),
        F.avg("airport_fee").alias("avg_airport_fee"),
        F.max("airport_fee").alias("max_airport_fee"),
        F.min("airport_fee").alias("min_airport_fee"),
        
        F.sum("congestion_surcharge").alias("total_congestion_surcharge"),
        F.avg("congestion_surcharge").alias("avg_congestion_surcharge"),
        F.max("congestion_surcharge").alias("max_congestion_surcharge"),
        F.min("congestion_surcharge").alias("min_congestion_surcharge"),
        
        F.sum("improvement_surcharge").alias("total_improvement_surcharge"),
        F.avg("improvement_surcharge").alias("avg_improvement_surcharge"),
        F.max("improvement_surcharge").alias("max_improvement_surcharge"),
        F.min("improvement_surcharge").alias("min_improvement_surcharge"),

        # DIMENSION 4: TRIP METRICS (DISTANCE & TIME)
        F.sum("trip_distance").alias("total_trip_distance"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.max("trip_distance").alias("max_trip_distance"),
        F.min("trip_distance").alias("min_trip_distance"),
        
        F.sum(F.expr("unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)")).alias("total_travel_time_seconds"),
        F.avg(F.expr("unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)")).alias("avg_travel_time_seconds"),
        F.max(F.expr("unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)")).alias("max_travel_time_seconds"),
        F.min(F.expr("unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)")).alias("min_travel_time_seconds")
    )
    
    # 5. Strict chronological sorting (Day -> Hour)
    df_final = df_aggregated.orderBy("DAY_SLICE", "HOUR_SLICE")

    return df_final  
        


In [0]:
df_gold_monthly = build_gold_agg(source_table=SILVER_TABLE, 
                                 target_table=GOLD_TABLE_MONTHLY, 
                                 time_slice="month") \
                                     .orderBy("TIME_SLICE")

display(df_gold_monthly.withColumnRenamed("TIME_SLICE", "MONTH"))

In [0]:
df_gold_daily = build_gold_agg(source_table=SILVER_TABLE, 
                                 target_table=GOLD_TABLE_DAILY, 
                                 time_slice="day") \
                                     .orderBy("TIME_SLICE")

display(df_gold_daily.withColumnRenamed("TIME_SLICE", "DAY"))

In [0]:
df_gold_hourly = build_gold_agg(source_table=SILVER_TABLE, 
                                 target_table=GOLD_TABLE_HOURLY, 
                                 time_slice="hour") \
                                     .orderBy("TIME_SLICE")

display(df_gold_daily.withColumnRenamed("TIME_SLICE", "HOUR"))

In [0]:
df_gold_may_analytics= build_gold_may_analytics(source_table=SILVER_TABLE, target_table=GOLD_TABLE_MAY_ANALYTICS)
display(df_gold_may_analytics)

#### Populating Gold Tables
* Submit gold tables for analytics


In [0]:
AUDIT_COLUMNS["CURRENT_SCRIPT"] = AUDIT_COLUMNS["AUD_ING_SLV_TO_GLD_SCRIPT"]

ingest_data({GOLD_TABLE_MONTHLY : df_gold_monthly}, GOLD_TABLE_MONTHLY, AUDIT_COLUMNS)
ingest_data({GOLD_TABLE_DAILY : df_gold_daily}, GOLD_TABLE_DAILY, AUDIT_COLUMNS)
ingest_data({GOLD_TABLE_HOURLY : df_gold_hourly}, GOLD_TABLE_HOURLY, AUDIT_COLUMNS)
ingest_data({GOLD_TABLE_MAY_ANALYTICS : df_gold_may_analytics}, GOLD_TABLE_MAY_ANALYTICS, AUDIT_COLUMNS)




In [0]:
%sql

SELECT * FROM ifood.gold.GLD_TXI_YEL_TRIP_MONTHLY;

In [0]:
%sql
SELECT * FROM ifood.gold.GLD_TXI_YEL_TRIP_DAILY;

In [0]:
%sql
SELECT * FROM ifood.gold.GLD_TXI_YEL_TRIP_HOURLY;

In [0]:
%sql
SELECT * FROM ifood.gold.GLD_TXI_YEL_TRIP_MAY_ANALYTICS

In [0]:
%sql
OPTIMIZE ifood.silver.SIL_TXI_YEL_TRIP
ZORDER BY (tpep_pickup_datetime);

OPTIMIZE ifood.gold.GLD_TXI_YEL_TRIP_MONTHLY
ZORDER BY (TIME_SLICE);

OPTIMIZE ifood.gold.GLD_TXI_YEL_TRIP_DAILY
ZORDER BY (TIME_SLICE);

OPTIMIZE ifood.gold.GLD_TXI_YEL_TRIP_HOURLY
ZORDER BY (TIME_SLICE);

OPTIMIZE ifood.gold.GLD_TXI_YEL_TRIP_MAY_ANALYTICS
ZORDER BY (DAY_SLICE, HOUR_SLICE);